<a href="https://colab.research.google.com/github/oswram19/parcial4_oswaldoramirez_1764382012/blob/main/ventas_transacciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
import warnings
# Configurar para ignorar advertencias de depreciación para una salida más limpia
warnings.filterwarnings('ignore', category=DeprecationWarning)

## cargar **achivo**

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/oswram19/parcial4_oswaldoramirez_1764382012/refs/heads/main/data/clave_D_asociacion.csv")
df.head()

,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,D-T0001,D-C0015,2026-04-02,Bebidas,Chocolate,1,Telefono
1,D-T0001,D-C0015,2026-04-02,Extras,Leche_extra,1,Telefono
2,D-T0001,D-C0015,2026-04-02,Salados,Wrap,1,Telefono
3,D-T0002,D-C0079,2026-01-21,Bebidas,Cafe_americano,1,Web
4,D-T0002,D-C0079,2026-01-21,Salados,Ensalada,1,Web


# Verificar valores nulos, duplicados y tipos de datos.

In [21]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nNumber of duplicate rows:")
print(df.duplicated().sum())
print("\nData types and non-null values:")
df.info()

Missing values per column:
transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             1
dtype: int64

Number of duplicate rows:
1

Data types and non-null values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 607 entries, 0 to 606
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  607 non-null    object
 1   cliente_id      607 non-null    object
 2   fecha           607 non-null    object
 3   categoria       607 non-null    object
 4   item            607 non-null    object
 5   cantidad        607 non-null    int64 
 6   canal           606 non-null    object
dtypes: int64(1), object(6)
memory usage: 33.3+ KB


# Preparar los datos en formato adecuado para aplicar reglas de asociación.

In [5]:
# Eliminar filas con valores nulos (solo 1 en 'canal')
df_cleaned = df.dropna()

# Eliminar filas duplicadas
df_cleaned = df_cleaned.drop_duplicates()

print("Dimensiones del DataFrame después de limpiar valores nulos y duplicados:", df_cleaned.shape)

Dimensiones del DataFrame después de limpiar valores nulos y duplicados: (605, 7)


In [6]:
# Convertir la columna 'item' en una lista de ítems por transacción
transactions = df_cleaned.groupby('transaccion_id')['item'].apply(list).reset_index()

display(transactions.head())

,transaccion_id,item
0,D-T0001,"[Chocolate, Leche_extra, Wrap]"
1,D-T0002,"[Cafe_americano, Ensalada, Latte]"
2,D-T0003,"[Capuchino, Chocolate, Croissant]"
3,D-T0004,"[Capuchino, Chocolate, Combo_tarde, Croissant,..."
4,D-T0005,[Leche_extra]


Ahora, transformaremos esta lista de ítems en un DataFrame codificado en one-hot, que es el formato de entrada para muchas librerías de minería de reglas de asociación como `mlxtend`.

In [7]:
from mlxtend.preprocessing import TransactionEncoder

# Obtener la lista de todas las transacciones (cada transacción es una lista de ítems)
list_of_transactions = transactions['item'].tolist()

te = TransactionEncoder()
te_ary = te.fit(list_of_transactions).transform(list_of_transactions)

df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
display(df_encoded.head())

,Cafe_americano,Capuchino,Chocolate,Combo_desayuno,Combo_estudiante,Combo_oficina,Combo_tarde,Crema_batida,Croissant,Dona,Ensalada,Galleta,Hielo,Latte,Leche_extra,Muffin,Panini,Sandwich,Sirope,Wrap
0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True
1,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False
2,False,True,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
3,False,True,True,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False


### Aplicar el algoritmo Apriori para encontrar conjuntos de ítems frecuentes

El algoritmo Apriori es utilizado para identificar los ítems que aparecen juntos con frecuencia en las transacciones. Se necesita definir un soporte mínimo (`min_support`) para considerar un conjunto de ítems como frecuente.

In [18]:
from mlxtend.frequent_patterns import apriori

# Calcular los conjuntos de ítems frecuentes
# min_support: el umbral mínimo para la frecuencia de un conjunto de ítems (0.01 significa que el ítem o conjunto debe aparecer en al menos el 1% de las transacciones)
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)

# Mostrar los primeros conjuntos de ítems frecuentes ordenados por soporte de forma descendente
display(frequent_itemsets.sort_values(by='support', ascending=False).head())

,support,itemsets
11,0.272222,(Galleta)
13,0.266667,(Latte)
0,0.266667,(Cafe_americano)
15,0.238889,(Muffin)
17,0.227778,(Sandwich)


In [17]:
from mlxtend.frequent_patterns import association_rules

# Generar las reglas de asociación
# metric: la métrica a usar para evaluar las reglas (e.g., 'lift', 'confidence', 'support')
# min_threshold: el umbral mínimo para la métrica seleccionada (un 'lift' > 1 indica una asociación positiva)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

# Ordenar las reglas por 'lift' de forma descendente y mostrar las primeras 10
# 'antecedents' son los ítems que aparecen primero, y 'consequents' son los ítems que tienden a aparecer junto con los antecedentes.
display(rules.sort_values(by="lift", ascending=False).head(10))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1281,"(Croissant, Hielo)","(Dona, Panini)",0.016667,0.011111,0.011111,0.666667,60.000000,1.0,0.010926,2.966667,1.000000,0.666667,0.662921,0.833333
1284,"(Dona, Panini)","(Croissant, Hielo)",0.011111,0.016667,0.011111,1.000000,60.000000,1.0,0.010926,inf,0.994382,0.666667,1.000000,0.833333
1343,"(Hielo, Latte, Sandwich)","(Combo_desayuno, Cafe_americano)",0.011111,0.038889,0.011111,1.000000,25.714286,1.0,0.010679,inf,0.971910,0.285714,1.000000,0.642857
1350,"(Combo_desayuno, Cafe_americano)","(Hielo, Latte, Sandwich)",0.038889,0.011111,0.011111,0.285714,25.714286,1.0,0.010679,1.384444,1.000000,0.285714,0.277689,0.642857
1337,"(Cafe_americano, Hielo, Latte)","(Combo_desayuno, Sandwich)",0.016667,0.027778,0.011111,0.666667,24.000000,1.0,0.010648,2.916667,0.974576,0.333333,0.657143,0.533333
1338,"(Cafe_americano, Sandwich, Latte)","(Combo_desayuno, Hielo)",0.016667,0.027778,0.011111,0.666667,24.000000,1.0,0.010648,2.916667,0.974576,0.333333,0.657143,0.533333
1356,"(Combo_desayuno, Sandwich)","(Cafe_americano, Hielo, Latte)",0.027778,0.016667,0.011111,0.400000,24.000000,1.0,0.010648,1.638889,0.985714,0.333333,0.389831,0.533333
1355,"(Combo_desayuno, Hielo)","(Cafe_americano, Sandwich, Latte)",0.027778,0.016667,0.011111,0.400000,24.000000,1.0,0.010648,1.638889,0.985714,0.333333,0.389831,0.533333
1352,"(Sandwich, Latte)","(Combo_desayuno, Cafe_americano, Hielo)",0.033333,0.016667,0.011111,0.333333,20.000000,1.0,0.010556,1.475000,0.982759,0.285714,0.322034,0.500000
1224,"(Combo_desayuno, Sirope)","(Latte, Leche_extra)",0.016667,0.033333,0.011111,0.666667,20.000000,1.0,0.010556,2.900000,0.966102,0.285714,0.655172,0.500000


### Generar reglas de asociación

Una vez que tenemos los conjuntos de ítems frecuentes, podemos generar reglas de asociación a partir de ellos. Estas reglas nos mostrarán qué ítems tienden a ser comprados juntos. Utilizaremos la métrica 'lift' para evaluar la fuerza de la asociación, donde un valor de lift mayor que 1 indica una asociación positiva.

In [16]:
from mlxtend.frequent_patterns import association_rules

# Generar las reglas de asociación
# metric: la métrica a usar para evaluar las reglas (e.g., 'lift', 'confidence', 'support')
# min_threshold: el umbral mínimo para la métrica seleccionada (un 'lift' > 1 indica una asociación positiva)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

# Ordenar las reglas por 'lift' de forma descendente y mostrar las primeras 10
# 'antecedents' son los ítems que aparecen primero, y 'consequents' son los ítems que tienden a aparecer junto con los antecedentes.
display(rules.sort_values(by="lift", ascending=False).head(10))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1281,"(Croissant, Hielo)","(Dona, Panini)",0.016667,0.011111,0.011111,0.666667,60.000000,1.0,0.010926,2.966667,1.000000,0.666667,0.662921,0.833333
1284,"(Dona, Panini)","(Croissant, Hielo)",0.011111,0.016667,0.011111,1.000000,60.000000,1.0,0.010926,inf,0.994382,0.666667,1.000000,0.833333
1343,"(Hielo, Latte, Sandwich)","(Combo_desayuno, Cafe_americano)",0.011111,0.038889,0.011111,1.000000,25.714286,1.0,0.010679,inf,0.971910,0.285714,1.000000,0.642857
1350,"(Combo_desayuno, Cafe_americano)","(Hielo, Latte, Sandwich)",0.038889,0.011111,0.011111,0.285714,25.714286,1.0,0.010679,1.384444,1.000000,0.285714,0.277689,0.642857
1337,"(Cafe_americano, Hielo, Latte)","(Combo_desayuno, Sandwich)",0.016667,0.027778,0.011111,0.666667,24.000000,1.0,0.010648,2.916667,0.974576,0.333333,0.657143,0.533333
1338,"(Cafe_americano, Sandwich, Latte)","(Combo_desayuno, Hielo)",0.016667,0.027778,0.011111,0.666667,24.000000,1.0,0.010648,2.916667,0.974576,0.333333,0.657143,0.533333
1356,"(Combo_desayuno, Sandwich)","(Cafe_americano, Hielo, Latte)",0.027778,0.016667,0.011111,0.400000,24.000000,1.0,0.010648,1.638889,0.985714,0.333333,0.389831,0.533333
1355,"(Combo_desayuno, Hielo)","(Cafe_americano, Sandwich, Latte)",0.027778,0.016667,0.011111,0.400000,24.000000,1.0,0.010648,1.638889,0.985714,0.333333,0.389831,0.533333
1352,"(Sandwich, Latte)","(Combo_desayuno, Cafe_americano, Hielo)",0.033333,0.016667,0.011111,0.333333,20.000000,1.0,0.010556,1.475000,0.982759,0.285714,0.322034,0.500000
1224,"(Combo_desayuno, Sirope)","(Latte, Leche_extra)",0.016667,0.033333,0.011111,0.666667,20.000000,1.0,0.010556,2.900000,0.966102,0.285714,0.655172,0.500000


In [15]:
# Mostrar las reglas más importantes filtrando por métricas clave
# Seleccionamos las columnas de interés: antecedentes, consecuentes, soporte, confianza y lift
reglas_principales = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

# Ordenamos por 'lift' para ver las asociaciones más fuertes
display(reglas_principales.sort_values(by='lift', ascending=False).head(10))

,antecedents,consequents,support,confidence,lift
1281,"(Croissant, Hielo)","(Dona, Panini)",0.011111,0.666667,60.000000
1284,"(Dona, Panini)","(Croissant, Hielo)",0.011111,1.000000,60.000000
1343,"(Hielo, Latte, Sandwich)","(Combo_desayuno, Cafe_americano)",0.011111,1.000000,25.714286
1350,"(Combo_desayuno, Cafe_americano)","(Hielo, Latte, Sandwich)",0.011111,0.285714,25.714286
1337,"(Cafe_americano, Hielo, Latte)","(Combo_desayuno, Sandwich)",0.011111,0.666667,24.000000
1338,"(Cafe_americano, Sandwich, Latte)","(Combo_desayuno, Hielo)",0.011111,0.666667,24.000000
1356,"(Combo_desayuno, Sandwich)","(Cafe_americano, Hielo, Latte)",0.011111,0.400000,24.000000
1355,"(Combo_desayuno, Hielo)","(Cafe_americano, Sandwich, Latte)",0.011111,0.400000,24.000000
1352,"(Sandwich, Latte)","(Combo_desayuno, Cafe_americano, Hielo)",0.011111,0.333333,20.000000
1224,"(Combo_desayuno, Sirope)","(Latte, Leche_extra)",0.011111,0.666667,20.000000


## Seleccionar y mostrar las 10 reglas con mayor lift (relevancia)

In [19]:

top_10_reglas = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False).head(10)
display(top_10_reglas)

,antecedents,consequents,support,confidence,lift
1281,"(Croissant, Hielo)","(Dona, Panini)",0.011111,0.666667,60.000000
1284,"(Dona, Panini)","(Croissant, Hielo)",0.011111,1.000000,60.000000
1343,"(Hielo, Latte, Sandwich)","(Combo_desayuno, Cafe_americano)",0.011111,1.000000,25.714286
1350,"(Combo_desayuno, Cafe_americano)","(Hielo, Latte, Sandwich)",0.011111,0.285714,25.714286
1337,"(Cafe_americano, Hielo, Latte)","(Combo_desayuno, Sandwich)",0.011111,0.666667,24.000000
1338,"(Cafe_americano, Sandwich, Latte)","(Combo_desayuno, Hielo)",0.011111,0.666667,24.000000
1356,"(Combo_desayuno, Sandwich)","(Cafe_americano, Hielo, Latte)",0.011111,0.400000,24.000000
1355,"(Combo_desayuno, Hielo)","(Cafe_americano, Sandwich, Latte)",0.011111,0.400000,24.000000
1352,"(Sandwich, Latte)","(Combo_desayuno, Cafe_americano, Hielo)",0.011111,0.333333,20.000000
1224,"(Combo_desayuno, Sirope)","(Latte, Leche_extra)",0.011111,0.666667,20.000000


### Interpretación de Reglas de Negocio

Analizando las reglas con mayor **Lift**, podemos extraer las siguientes conclusiones estratégicas:

1. **El Combo Energético (Sandwich + Latte → Café Americano):**
   - *Interpretación:* Existe una probabilidad extremadamente alta (Lift ~24) de que un cliente que pida un Sandwich y un Latte también consuma un Café Americano.
   - *Acción:* Ideal para crear un 'Super Menú' que incluya los tres productos a un precio especial.

2. **Preferencia por Bebidas Frías (Hielo + Latte → Café Americano):**
   - *Interpretación:* Los clientes que personalizan su bebida con hielo y piden un Latte muestran una fuerte asociación con el Café Americano.
   - *Acción:* Sugiere un perfil de cliente que consume múltiples bebidas; se podría ofrecer una segunda bebida a mitad de precio en pedidos de este tipo.

3. **La Pareja de la Tarde (Dona → Panini):**
   - *Interpretación:* Quienes compran una Dona tienen una altísima probabilidad de llevar también un Panini.
   - *Acción:* Colocar las Donas cerca de la vitrina de Paninis para fomentar la compra por impulso de algo dulce tras lo salado.

4. **Complementos de Desayuno (Combo Desayuno + Café Americano → Hielo):**
   - *Interpretación:* Los clientes del Combo Desayuno que toman café suelen pedirlo con hielo.
   - *Acción:* Asegurar siempre el stock de hielo en horas pico de la mañana y ofrecer vasos más grandes para café helado.

5. **Personalización Crítica (Leche Extra → Latte):**
   - *Interpretación:* Hay una asociación perfecta (Lift 20) entre pedir Leche Extra y el consumo de Latte.
   - *Acción:* Capacitar al personal para ofrecer 'Leche Extra' o variedades de leche (almendra, soya) cada vez que se ordene un Latte, ya que es un valor añadido esperado por este segmento.

### Recomendaciones Comerciales Estratégicas

1. **Diseño de Menús Dinámicos (Bundling):**
   - Dado el altísimo *Lift* entre el **Sandwich, Latte y Café Americano**, se recomienda lanzar un "Combo Ejecutivo Premium". Empaquetar estos productos bajo un solo precio ligeramente descontado puede aumentar el ticket promedio al incentivar la compra de la tercera categoría que el cliente ya tiende a consumir.

2. **Optimización de Punto de Venta (Layout):**
   - Existe una fuerte correlación entre las **Donas y los Paninis**. Se sugiere ubicar las estaciones de productos dulces (repostería) inmediatamente después de la zona de salados en el flujo de la tienda o en el menú digital. Esto aprovecha el impulso de compra de un postre tras consumir un plato fuerte.

3. **Campaña de Up-selling en Personalización:**
   - La regla **Leche Extra → Latte** indica que el cliente de Latte valora la personalización. Se recomienda entrenar al equipo de ventas para ofrecer proactivamente variedades de leche premium (almendra, avena) o extras adicionales cada vez que se pida un Latte, aumentando así el margen de ganancia por bebida preparada.